# Weekly Trend Watchlist Lab (No Recent MomLE)

This notebook shows symbols that are in a **strong weekly uptrend** but had **no weekly `MomLE` in the last 2 weekly bars** (about 10 trading days).

- Source signals: `out/indicators/momentum_tv_match/weekly/*.csv`
- Optional shortlist cache: `out/reports/momentum/weekly_trend_no_recent_momle_10d.csv`
- Plot window: last 3 calendar years
- Max symbols plotted per run: `24` (change `MAX_SYMBOLS` in code cell)


In [ ]:
from pathlib import Path
import csv
import math
import sys
from datetime import date, timedelta

import matplotlib.dates as mdates
import matplotlib.pyplot as plt

OUT_ROOT = Path("out")

RECENT_WINDOW_BARS = 2  # weekly bar window (~10 trading days)
YEARS_TO_PLOT = 3
MAX_SYMBOLS = 24
CHART_STYLE = "candles"  # "candles" or "line"
MOMENTUM_DIR = OUT_ROOT / "indicators" / "momentum_tv_match" / "weekly"
SHORTLIST_CSV = OUT_ROOT / "reports" / "momentum" / "weekly_trend_no_recent_momle_10d.csv"


def resolve_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "watchlist.txt").exists() and (candidate / OUT_ROOT).exists():
            return candidate
    return Path.cwd()


def parse_float(value: str | None) -> float | None:
    raw = (value or "").strip()
    if not raw:
        return None
    return float(raw)


def read_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def compute_ema(values: list[float], period: int) -> list[float | None]:
    out: list[float | None] = [None] * len(values)
    if period <= 0 or len(values) < period:
        return out
    alpha = 2.0 / (period + 1.0)
    seed = sum(values[:period]) / period
    out[period - 1] = seed
    prev = seed
    for idx in range(period, len(values)):
        current = (values[idx] - prev) * alpha + prev
        out[idx] = current
        prev = current
    return out


def scan_trend_candidates() -> list[dict[str, str]]:
    results: list[dict[str, str]] = []
    for csv_path in sorted(MOMENTUM_PATH.glob("*.csv")):
        rows = read_rows(csv_path)
        if len(rows) < 200:
            continue

        dates: list[date] = []
        closes: list[float] = []
        events: list[str] = []
        valid = True
        for row in rows:
            close = parse_float(row.get("Close"))
            if close is None:
                valid = False
                break
            dates.append(date.fromisoformat(row["Date"]))
            closes.append(close)
            events.append((row.get("Event") or "").strip())
        if not valid or len(closes) < 200:
            continue

        recent_events = events[-RECENT_WINDOW_BARS:]
        if any(event == "MomLE" for event in recent_events):
            continue

        last_momle_idx = None
        for idx in range(len(events) - 1, -1, -1):
            if events[idx] == "MomLE":
                last_momle_idx = idx
                break
        if last_momle_idx is None:
            continue

        ema50 = compute_ema(closes, 50)
        ema200 = compute_ema(closes, 200)
        latest_idx = len(closes) - 1
        latest_ema50 = ema50[latest_idx]
        latest_ema200 = ema200[latest_idx]
        if latest_ema50 is None or latest_ema200 is None:
            continue
        if latest_idx == 0 or ema50[latest_idx - 1] is None:
            continue

        close = closes[latest_idx]
        ema50_up = latest_ema50 > float(ema50[latest_idx - 1])
        if not (close > latest_ema50 and latest_ema50 > latest_ema200 and ema50_up):
            continue

        bars_since = latest_idx - last_momle_idx
        results.append(
            {
                "Rank": "",
                "Symbol": csv_path.stem,
                "Score": "",
                "LatestDate": dates[latest_idx].isoformat(),
                "LastMomLEDate": dates[last_momle_idx].isoformat(),
                "BarsSinceLastMomLE": str(bars_since),
            }
        )

    return results


ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.plotting.candles import (
    build_ohlc_arrays,
    overlay_ema,
    overlay_event_markers,
    plot_candlesticks,
    plot_volume,
)

MOMENTUM_PATH = ROOT / MOMENTUM_DIR
SHORTLIST_PATH = ROOT / SHORTLIST_CSV


def load_symbols() -> list[dict[str, str]]:
    if SHORTLIST_PATH.exists():
        rows = read_rows(SHORTLIST_PATH)
        if rows:
            def rank_key(row: dict[str, str]) -> int:
                raw = (row.get("Rank") or "").strip()
                return int(raw) if raw.isdigit() else 10_000

            return sorted(rows, key=rank_key)
    return scan_trend_candidates()


symbols = load_symbols()

print(f"Repo root: {ROOT}")
print(f"Momentum dir: {MOMENTUM_PATH.relative_to(ROOT)}")
print(f"Recent-window bars (weekly): {RECENT_WINDOW_BARS}")
print(f"Symbols in strong weekly trend with no recent MomLE (~10d): {len(symbols)}")
if MAX_SYMBOLS is not None and MAX_SYMBOLS > 0:
    symbols = symbols[:MAX_SYMBOLS]
print(f"Symbols plotted: {len(symbols)}")
print("Symbols:", ", ".join(row["Symbol"] for row in symbols) if symbols else "(none)")

if not symbols:
    raise RuntimeError("No symbols found for continuation watchlist.")

n = len(symbols)
cols = 2 if n > 1 else 1
rows_count = math.ceil(n / cols)
fig, axes = plt.subplots(
    rows_count * 2,
    cols,
    figsize=(8.8 * cols, 4.8 * rows_count),
    squeeze=False,
    gridspec_kw={"height_ratios": [3, 1] * rows_count},
)

for idx, info in enumerate(symbols):
    grid_row = idx // cols
    grid_col = idx % cols
    ax_price = axes[grid_row * 2][grid_col]
    ax_volume = axes[grid_row * 2 + 1][grid_col]

    symbol = info["Symbol"]
    csv_path = MOMENTUM_PATH / f"{symbol}.csv"
    rows = read_rows(csv_path)

    ohlc = build_ohlc_arrays(rows)
    dates = ohlc["dates"]
    opens = ohlc["opens"]
    highs = ohlc["highs"]
    lows = ohlc["lows"]
    closes = ohlc["closes"]
    volumes = ohlc["volumes"]

    events_by_date: dict[date, str] = {}
    for row in rows:
        raw_day = (row.get("Date") or "").strip()
        if not raw_day:
            continue
        try:
            day = date.fromisoformat(raw_day)
        except ValueError:
            continue
        events_by_date[day] = (row.get("Event") or "").strip()

    if not dates:
        ax_price.set_title(f"{symbol} (no data)")
        ax_price.axis("off")
        ax_volume.axis("off")
        continue

    ema50 = compute_ema(closes, 50)
    ema200 = compute_ema(closes, 200)
    events = [events_by_date.get(day, "") for day in dates]

    start_date = dates[-1] - timedelta(days=365 * YEARS_TO_PLOT)
    start_idx = next((i for i, day in enumerate(dates) if day >= start_date), 0)

    plot_dates = dates[start_idx:]
    plot_opens = opens[start_idx:]
    plot_highs = highs[start_idx:]
    plot_lows = lows[start_idx:]
    plot_closes = closes[start_idx:]
    plot_volumes = volumes[start_idx:]
    plot_ema50 = ema50[start_idx:]
    plot_ema200 = ema200[start_idx:]
    plot_events = events[start_idx:]

    if CHART_STYLE == "candles":
        plot_candlesticks(
            ax_price,
            plot_dates,
            plot_opens,
            plot_highs,
            plot_lows,
            plot_closes,
            width_days=6.0,
        )
    elif CHART_STYLE == "line":
        ax_price.plot(plot_dates, plot_closes, label="Close", color="#1f77b4", linewidth=1.2)
    else:
        raise ValueError(f"Unsupported CHART_STYLE: {CHART_STYLE}")

    overlay_ema(ax_price, plot_dates, plot_ema50, label="EMA 50", color="#ff7f0e", linewidth=1.0)
    overlay_ema(ax_price, plot_dates, plot_ema200, label="EMA 200", color="#2ca02c", linewidth=1.0)

    overlay_event_markers(
        ax_price,
        plot_dates,
        plot_closes,
        plot_events,
        marker_map={
            "MomLE": {"label": "MomLE", "marker": "^", "size": 26, "color": "#0a84ff", "alpha": 0.75},
            "MomSE": {"label": "MomSE", "marker": "v", "size": 24, "color": "#d81b60", "alpha": 0.45},
        },
    )

    last_momle_date_raw = (info.get("LastMomLEDate") or "").strip()
    if last_momle_date_raw:
        last_momle_date = date.fromisoformat(last_momle_date_raw)
        for day, price, event in zip(plot_dates, plot_closes, plot_events, strict=True):
            if day == last_momle_date and event == "MomLE":
                ax_price.scatter(
                    [day],
                    [price],
                    marker="o",
                    s=90,
                    facecolors="none",
                    edgecolors="#ffd166",
                    linewidths=1.8,
                    label="Last MomLE",
                    zorder=8,
                )
                break

    rank = (info.get("Rank") or "").strip()
    score = (info.get("Score") or "").strip()
    bars_since = (info.get("BarsSinceLastMomLE") or "").strip()
    title_prefix = f"#{rank} " if rank else ""
    score_text = f" | score: {score}" if score else ""
    bars_text = f" | bars since: {bars_since}" if bars_since else ""
    ax_price.set_title(f"{title_prefix}{symbol}{score_text} | last MomLE: {last_momle_date_raw}{bars_text}")
    ax_price.grid(alpha=0.25)
    ax_price.legend(loc="best", fontsize=8)
    ax_price.tick_params(axis="x", labelbottom=False)

    plot_volume(
        ax_volume,
        plot_dates,
        plot_opens,
        plot_closes,
        plot_volumes,
        width_days=6.0,
    )
    ax_volume.set_ylabel("Vol", fontsize=8)
    ax_volume.grid(alpha=0.18)
    ax_volume.xaxis.set_major_locator(mdates.YearLocator())
    ax_volume.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

for idx in range(len(symbols), rows_count * cols):
    grid_row = idx // cols
    grid_col = idx % cols
    axes[grid_row * 2][grid_col].axis("off")
    axes[grid_row * 2 + 1][grid_col].axis("off")

fig.suptitle("Weekly Continuation Watchlist: Strong Trend, No Recent MomLE", fontsize=14)
fig.tight_layout()
plt.show()

